# Sesión 01: Capa de Recolección y Scraping Directo Local (OSINT)

Este notebook representa la **Mesa de Control de la Sesión 01** del proyecto **SAIEL** (Sistema de Análisis de Inteligencia Electoral Local).

### 🎯 Objetivos de la Sesión
1. Validar la recolección de comentarios de candidatos para la presidencia municipal de **Tepic, Nayarit**.
2. Ejecutar y testear el colector híbrido `SocialCollector` de forma local (Bunker Mode).
3. Comprobar la resiliencia del pipeline (Scraping directo local y fallback sintético con `Faker`).
4. Visualizar los datos crudos recolectados antes de su envío a la capa de ingesta.

## 1. Configuración del Entorno e Importaciones

Cargamos las variables de entorno relativas desde el archivo `.env` y configuramos el `PYTHONPATH` para importar el módulo de producción `src/collectors/social_collector.py` sin conflictos de directorios.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# Configurar la ruta base del proyecto dinámicamente
base_path = Path(os.getcwd()).resolve().parent
sys.path.append(str(base_path))

# Cargar variables de entorno
load_dotenv(base_path / ".env")
print(f"✓ Ruta Base de Trabajo establecida: {base_path}")

## 2. Inicialización del Colector Híbrido

Instanciamos la clase `SocialCollector` desarrollada en `src/collectors/social_collector.py` pasando la ruta dinámica calculada.

In [ ]:
from src.collectors.social_collector import SocialCollector

collector = SocialCollector(base_path=base_path)
print("✓ SocialCollector inicializado y listo en modo local/híbrido.")

## 3. Simulación de Recolección de Datos de Candidatos (Tepic, Nayarit)

Corremos la recolección local para una muestra de candidatos en Tepic (ej. Geraldine Ponce, Adahan Casas, Ivideliza Reyes). 

El colector intentará un raspado web directo ligero y generará un dataset realista usando contextos políticos locales si se activa el modo local o no hay internet/credenciales.

In [ ]:
# Definimos objetivos de recolección en Tepic
candidatos = [
    {"nombre": "Geraldine Ponce", "plataforma": "instagram", "limite": 15},
    {"nombre": "Adahan Casas", "plataforma": "facebook", "limite": 10},
    {"nombre": "Ivideliza Reyes", "plataforma": "instagram", "limite": 12}
]

resultados_totales = {}

for cand in candidatos:
    print(f"\nProcesando candidato: {cand['nombre']}")
    data = collector.collect_social_data(
        target=cand['nombre'],
        platform=cand['plataforma'],
        limit=cand['limite'],
        mode="local"  # Forzamos modo local para testeo rápido a costo cero
    )
    resultados_totales[cand['nombre']] = data
    print(f" -> Colectados {len(data)} comentarios exitosamente.")

## 4. Visualización y Control de Calidad de Datos (EDA)

Convertimos el lote recolectado de la candidata "Geraldine Ponce" a un DataFrame de Pandas para analizar la estructura, la calidad de los comentarios y comprobar que los datos se alinean con las directrices de la arquitectura.

In [ ]:
df_geraldine = pd.DataFrame(resultados_totales["Geraldine Ponce"])

print("=== Muestra de Datos Crudos de Geraldine Ponce ===")
display(df_geraldine.head(5))

print("\n=== Distribución de Likes de Comentarios ===")
print(df_geraldine['likes'].describe())

## 5. Validación Física de Archivos en Disco (Trazabilidad)

Comprobamos que los archivos JSON se hayan persistido físicamente en `data/raw/` bajo la nomenclatura estándar recomendada por el plan de trazabilidad.

In [ ]:
print("=== Archivos Crudos Persistidos en data/raw/ ===")
raw_path = base_path / "data" / "raw"
for f in raw_path.glob("raw_*.json"):
    size_kb = f.stat().st_size / 1024
    mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    print(f" - {f.name} ({size_kb:.2f} KB) | Modificado: {mtime}")

## Conclusiones de la Sesión 01
1. El colector híbrido opera a la perfección en **Bunker Mode**, abstrayendo la dependencia de API Keys y créditos durante el desarrollo local.
2. Los archivos crudos se almacenan en rutas relativas correctas (`data/raw/`) respetando Separation of Concerns.
3. El dataset cuenta con campos alfanuméricos realistas y limpios listos para alimentar a la capa de Ingesta y Anonimización (Sesión 02).